In [ ]:
%%configure -f
{
    "conf": {
        "spark.dynamicAllocation.enabled": "false",
        "spark.driver.cores": "4",
        "spark.driver.memory": "28g",
        "spark.executor.cores": "4",
        "spark.executor.memory": "28g",
        "spark.executor.instances": 1
    }
}

# Feature Engineering

**Time Series Forecasting Accelerator**

| Parameter | Value |
|-----------|-------|
| Scenario | supermarket_net_sales_forecast |
| Generated | 2026-06-03 |
| Date Column | WEEK_START_DT |
| Target Column | TOTAL_NET_SALES |
| ID Column | STORE_LOCATION_ID |
| Time Granularity | Weekly (Thursday start) |
| Forecast Horizon | 4 weeks |

This notebook performs feature engineering for the supermarket net sales forecasting pipeline:

1. **Data Type Classification** â€” static, numeric, binary, categorical columns
2. **Feature Engineering** â€” lags (4,8,13,26,52), rolling statistics, calendar features
3. **Feature Inspection** â€” correlation with target, XGBoost feature importance
4. **Per-Cluster Analysis** â€” separate feature importance by profile_cluster

**Customizations applied:**
- Lags: [4, 8, 13, 26, 52] (minimum lag=4 for 4-week forecast horizon)
- Rolling windows: RollingMean/Std with windows 4, 13, 52
- Holiday/event flags as direct features (known in advance)
- Time-varying regressors excluded (not available at forecast time)
- Collinear columns removed: BASELINE_WEEKLY_SALES, TOTAL_TRANSACTIONS, TOTAL_GROSS_MARGIN

# Import Required Libraries

In [ ]:
# Data elaboration
import pandas as pd
import numpy as np
import datetime as dt
from pathlib import Path

# Plotting
import matplotlib.pyplot as plt
%matplotlib inline

# Nixtla MLForecast
from mlforecast import MLForecast
from mlforecast.lag_transforms import RollingMean, RollingStd

# Model
from xgboost import XGBRegressor

print("\u2705 All imports successful!")

# Configuration

Configuration parameters for the time series analysis.

In [ ]:
# Time series configuration
date_var = 'WEEK_START_DT'
date_format = '%Y-%m-%d'
y = 'TOTAL_NET_SALES'
unique_id = 'STORE_LOCATION_ID'
frequency = 'W-THU'

# Fabric Lakehouse configuration
LAKEHOUSE_NAME = "ts_mmm"
INPUT_TABLE = "supermarket_net_sales_forecast_clustered"
OUTPUT_TABLE = "supermarket_net_sales_forecast_features"

# Columns to exclude (collinear with target â€” confirmed in Phase 4.2 CP-0005)
EXCLUDE_COLS = ['BASELINE_WEEKLY_SALES', 'TOTAL_TRANSACTIONS', 'TOTAL_GROSS_MARGIN']

root_path = Path.cwd().parent.parent

print("\u2705 Configuration loaded")
print(f"   date_var: {date_var}")
print(f"   unique_id: {unique_id}")
print(f"   target: {y}")
print(f"   frequency: {frequency}")
print(f"   input_table: {INPUT_TABLE}")
print(f"   output_table: {OUTPUT_TABLE}")
print(f"   excluded_cols: {EXCLUDE_COLS}")

# Load Data from Lakehouse

Load the clustered data (output of Notebook 04) which already contains the `profile_cluster` column.

In [ ]:
# Load data from Lakehouse â€” the clustered table already has profile_cluster column
try:
    df = spark.table(INPUT_TABLE)
    df = df.toPandas()
    print(f"\u2705 Loaded {len(df)} rows from Lakehouse table: {INPUT_TABLE}")
except Exception as e:
    print(f"\u274c Error loading from {INPUT_TABLE}: {e}")
    print("Attempting fallback to local file...")
    df = pd.read_parquet(f"{root_path}/data/{INPUT_TABLE}.parquet")
    print(f"\u2705 Loaded {len(df)} rows from local file")

print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)[:15]}...")
print(f"profile_cluster values: {df['profile_cluster'].unique().tolist()}")

In [ ]:
# Remove collinear columns (confirmed in Phase 4.2 â€” CP-0005)
cols_before = df.shape[1]
df = df.drop(columns=[c for c in EXCLUDE_COLS if c in df.columns], errors='ignore')
print(f"\u2705 Removed {cols_before - df.shape[1]} collinear columns: {EXCLUDE_COLS}")
print(f"   Shape after exclusion: {df.shape}")

# Standardize Column Names

Rename date column to match mlforecast expectations and extract calendar features.

In [ ]:
# Standardize column names â€” rename date column to 'ds' for mlforecast
rename_map = {
    date_var: "ds",  # mlforecast expects time column as 'ds'
}
df = df.rename(columns=rename_map)
date_var = 'ds'

# Ensure proper datetime type
df['ds'] = pd.to_datetime(df['ds'])

# Extract date features (calendar features known in advance)
df['year'] = df['ds'].dt.year
df['month'] = df['ds'].dt.month
df['week'] = df['ds'].dt.isocalendar().week.astype(int)
date_features = ["year", "month", "week"]

print(f"\u2705 Column renamed: WEEK_START_DT \u2192 ds")
print(f"\u2705 Date features extracted: {date_features}")
print(f"   Date range: {df['ds'].min()} to {df['ds'].max()}")
print(f"   Shape: {df.shape}")

# Ensure Proper Data Types

Classify columns into static, numeric, binary, and categorical types.

## Static features (constant within each store)

In [ ]:
# Identify static columns (constant within each unique_id)
def get_static_cols(df, id_col, exclude_cols=None):
    """Returns columns that are constant within each unique_id (static covariates)."""
    if exclude_cols is None:
        exclude_cols = []
    candidate_cols = [c for c in df.columns if c not in [id_col, 'ds'] + exclude_cols]
    static = []
    non_static = []
    for col in candidate_cols:
        n_unique_per_id = df.groupby(id_col)[col].nunique()
        if (n_unique_per_id <= 1).all():
            static.append(col)
        else:
            non_static.append(col)
    return static, non_static

static_cols, non_static_cols = get_static_cols(df, unique_id, exclude_cols=[y] + date_features + ['profile_cluster'])

print(f"\u2705 Static columns ({len(static_cols)}):")
for c in static_cols:
    df[c] = df[c].astype("category")
    print(f"  - {c}")

print(f"\n\u26a0\ufe0f Non-static (time-varying) columns ({len(non_static_cols)}):")
for c in non_static_cols[:10]:
    print(f"  - {c}")
if len(non_static_cols) > 10:
    print(f"  ... and {len(non_static_cols) - 10} more")

## Binary features (0/1 flags)

In [ ]:
# Identify binary columns (0/1 flags â€” holidays and events)
def get_binary_cols(df, exclude_cols=None):
    """Returns columns that contain only binary values (0 and 1)."""
    if exclude_cols is None:
        exclude_cols = []
    binary_cols = []
    for col in df.columns:
        if col in exclude_cols:
            continue
        unique_vals = set(df[col].dropna().unique())
        if unique_vals.issubset({0, 1, 0.0, 1.0}):
            binary_cols.append(col)
    return binary_cols

binary_cols = get_binary_cols(df, exclude_cols=[unique_id, 'ds', y, 'profile_cluster'])
print(f"\u2705 Found {len(binary_cols)} binary columns:")
for c in binary_cols:
    print(f"  - {c}")

# Separate into holiday/event flags (known in advance) and other binary
holiday_event_flags = [c for c in binary_cols if c.startswith('HOLIDAY_') or c.startswith('EVENT_')]
other_binary = [c for c in binary_cols if c not in holiday_event_flags]
print(f"\n   Holiday/Event flags (direct features): {len(holiday_event_flags)}")
print(f"   Other binary (static competitor flags): {len(other_binary)}")

## Numeric features

In [ ]:
# Identify numeric columns (excluding binary, static, date features)
numeric_cols = df.select_dtypes(include=['float64', 'float32', 'int64']).columns.tolist()
# Remove columns that are binary, date features, or the unique_id
numeric_cols = [c for c in numeric_cols if c not in binary_cols + date_features + [unique_id]]

print(f"\u2705 Numeric columns ({len(numeric_cols)}):")
for c in numeric_cols[:15]:
    print(f"  - {c}")
if len(numeric_cols) > 15:
    print(f"  ... and {len(numeric_cols) - 15} more")

# Ensure numeric types
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

## Categorical features

In [ ]:
# Cast binary columns to category type
for col in binary_cols:
    if col in df.columns:
        df[col] = df[col].astype("category")

# Identify categorical columns (object/category/bool that are NOT static, NOT date-derived)
categorical_cols = [
    c for c in df.select_dtypes(include=["object", "category", "bool"]).columns
    if (c != unique_id) and (c not in date_features) and (c != 'profile_cluster')
]

print(f"\u2705 Binary columns cast to category: {len(binary_cols)}")
print(f"\u2705 All categorical columns ({len(categorical_cols)}):")
for c in categorical_cols[:10]:
    print(f"  - {c} | dtype={df[c].dtype} | nunique={df[c].nunique()}")
if len(categorical_cols) > 10:
    print(f"  ... and {len(categorical_cols) - 10} more")

print(f"\n\u2705 Dtypes summary:")
print(df.dtypes.value_counts())

# Infer Frequency

In [ ]:
# Infer frequency from the data
sorted_dates = sorted(df['ds'].unique())
dti = pd.DatetimeIndex(sorted_dates)
FREQ = pd.infer_freq(dti)
print(f"Inferred frequency: {FREQ}")

# If frequency inference fails, set manually
if FREQ is None:
    FREQ = 'W-THU'  # Weekly Thursday-start (confirmed in Phase 1)
    print(f"Frequency set manually to: {FREQ}")

# Verify
print(f"\nVerification:")
print(f"  All Thursdays: {all(d.strftime('%A') == 'Thursday' for d in sorted_dates)}")
print(f"  All 7-day diffs: {all((sorted_dates[i+1] - sorted_dates[i]).days == 7 for i in range(len(sorted_dates)-1))}")
print(f"  Sample dates: {sorted_dates[:3]}")

### \u2705 CHECK POINT\n",
Check inferred frequency with data scientist.\n",
\n",
- **Inferred frequency:** W-THU (Weekly, anchored on Thursday)\n",
- **All dates are Thursdays:** Yes\n",
- **All inter-date intervals:** 7 days\n",
- **Date range:** 2023-04-06 to 2025-03-27 (104 weeks)

# Feature Engineering with MLForecast

Generate lag features, rolling statistics, and calendar features using MLForecast.

**Parameters:**
- Lags: [4, 8, 13, 26, 52] (minimum lag=4 to avoid leakage with 4-week horizon)
- Rolling windows: Mean/Std with windows 4, 13, 52 (all anchored to lag 4)
- Calendar: week, month, quarter, year
- All features treated as dynamic (event flags change over time)

In [ ]:
# --- Feature Engineering Parameters ---
FORECAST_HORIZON = 4
LAGS = [4, 8, 13, 26, 52]

# CUSTOMIZED: All rolling transforms anchored to lag 4 to prevent data loss
# (Attaching RollingMean(52) to lag 52 would require 103 prior periods, leaving only 1 row/store)
lag_transforms = {
    4: [
        RollingMean(window_size=4),
        RollingStd(window_size=4),
        RollingMean(window_size=13),
        RollingStd(window_size=13),
        RollingMean(window_size=52),
        RollingStd(window_size=52),
    ]
}

print(f"âœ… Feature parameters configured")
print(f"   Forecast horizon: {FORECAST_HORIZON} weeks")
print(f"   Lags: {LAGS}")
print(f"   Rolling transforms: Mean/Std with windows [4, 13, 52] anchored to lag 4")
print(f"   Min lag ({min(LAGS)}) >= horizon ({FORECAST_HORIZON}) âœ“ No leakage")

In [ ]:
# --- MLForecast Preprocessing ---
# Drop static and categorical columns for MLForecast (it works with numeric data)
drop_for_mlf = static_cols + ['profile_cluster']
df_mlf = df.drop(columns=[c for c in drop_for_mlf if c in df.columns]).copy()

# Build MLForecast object
mlf = MLForecast(
    models=[],  # No models needed, just preprocessing
    freq=FREQ,
    lags=LAGS,
    lag_transforms=lag_transforms,
    date_features=['week', 'month', 'quarter', 'year'],
)

# Run preprocessing â€” generates lag/rolling features and drops rows with insufficient history
df_features = mlf.preprocess(
    df_mlf,
    id_col=unique_id,
    time_col='ds',
    target_col=y,
    static_features=[],  # All features are dynamic (event flags change over time)
)

print(f"âœ… MLForecast preprocessing complete")
print(f"   Input shape: {df_mlf.shape}")
print(f"   Output shape: {df_features.shape}")
print(f"   Rows per store: {df_features.shape[0] // df[unique_id].nunique()}")
print(f"   Features generated: {df_features.shape[1] - 3} (excluding unique_id, ds, target)")
print(f"\n   New columns (lag/rolling):")
new_cols = [c for c in df_features.columns if c not in df_mlf.columns]
for c in new_cols:
    print(f"     - {c}")

In [ ]:
# --- Join profile_cluster back for per-cluster analysis ---
cluster_map = df[[unique_id, 'ds', 'profile_cluster']].copy()
df_features = df_features.merge(cluster_map, on=[unique_id, 'ds'], how='left')

# Join profiling metrics (cv2, adi) if present
profiling_cols = ['cv2', 'adi', 'profile_class']
existing_prof = [c for c in profiling_cols if c in df.columns and c not in df_features.columns]
if existing_prof:
    prof_map = df[[unique_id, 'ds'] + existing_prof].copy()
    df_features = df_features.merge(prof_map, on=[unique_id, 'ds'], how='left')

print(f"âœ… Joined cluster and profiling columns")
print(f"   Final shape: {df_features.shape}")
print(f"   Clusters: {df_features['profile_cluster'].value_counts().to_dict()}")
print(f"   Null values: {df_features.isnull().sum().sum()}")

# Feature Correlation Analysis

Compute correlations between generated features and the target variable to identify the most predictive signals.

In [ ]:
# --- Correlation with Target ---
exclude_corr = {unique_id, 'ds', y, 'profile_cluster', 'profile_class'}
corr_cols = [c for c in df_features.columns if c not in exclude_corr and pd.api.types.is_numeric_dtype(df_features[c])]

correlations = df_features[corr_cols + [y]].corr()[y].drop(y).abs().sort_values(ascending=False)

print("=== Top 20 Features by Correlation with Target ===")
print(correlations.head(20).to_string())

# Visualize top correlations
fig, ax = plt.subplots(figsize=(10, 8))
correlations.head(20).plot(kind='barh', ax=ax)
ax.set_xlabel('Absolute Correlation')
ax.set_title(f'Top 20 Features by Correlation with {y}')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

# XGBoost Feature Importance

Train an XGBoost model to assess feature importance via gain-based splitting. This provides a non-linear importance measure complementary to correlation.

In [ ]:
# --- Global XGBoost Feature Importance ---
from xgboost import XGBRegressor

exclude_importance = {unique_id, 'ds', y, 'profile_cluster', 'profile_class'}
feature_cols = [c for c in df_features.columns if c not in exclude_importance]

X = df_features[feature_cols].copy()
y_vals = df_features[y].values

# Convert any remaining categoricals to numeric codes
for c in X.select_dtypes(include=['object', 'category']).columns:
    X[c] = X[c].astype('category').cat.codes

model = XGBRegressor(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42, n_jobs=-1)
model.fit(X, y_vals)

# Feature importance (gain-based)
importance = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)

print("=== Global Top 15 Feature Importance (XGBoost Gain) ===")
for feat, score in importance.head(15).items():
    print(f"  {feat}: {score:.4f}")

# Visualize
fig, ax = plt.subplots(figsize=(10, 8))
importance.head(15).plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('Feature Importance (Gain)')
ax.set_title('Global XGBoost Feature Importance â€” Top 15')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

# Per-Cluster Feature Importance

Train separate XGBoost models per `profile_cluster` to understand which features drive each cluster differently.

In [ ]:
# --- Per-Cluster XGBoost Importance ---
cluster_importance = {}
clusters = sorted(df_features['profile_cluster'].unique())

for cl in clusters:
    cl_df = df_features[df_features['profile_cluster'] == cl]
    X_cl = cl_df[feature_cols].copy()
    for c in X_cl.select_dtypes(include=['object', 'category']).columns:
        X_cl[c] = X_cl[c].astype('category').cat.codes
    y_cl = cl_df[y].values
    
    m = XGBRegressor(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42, n_jobs=-1)
    m.fit(X_cl, y_cl)
    imp_cl = pd.Series(m.feature_importances_, index=feature_cols).sort_values(ascending=False)
    cluster_importance[cl] = imp_cl

# Display per-cluster results
print("=== Per-Cluster Top 5 Features ===")
for cl in clusters:
    print(f"\n  {cl} ({len(df_features[df_features['profile_cluster'] == cl])} rows):")
    for feat, score in cluster_importance[cl].head(5).items():
        print(f"    {feat}: {score:.4f}")

# Visualize side-by-side
fig, axes = plt.subplots(1, len(clusters), figsize=(4*len(clusters), 6), sharey=False)
for i, cl in enumerate(clusters):
    ax = axes[i] if len(clusters) > 1 else axes
    cluster_importance[cl].head(10).plot(kind='barh', ax=ax, color=f'C{i}')
    ax.set_title(f'{cl}', fontsize=9)
    ax.set_xlabel('Importance')
    ax.invert_yaxis()
plt.suptitle('Per-Cluster Feature Importance (Top 10)', fontsize=12)
plt.tight_layout()
plt.show()

# Save Features to Lakehouse

Save the engineered features as Delta tables:
- **Global:** `supermarket_net_sales_forecast_features` (all clusters combined)
- **Per-cluster:** `supermarket_net_sales_forecast_features_cluster_{name}` (5 tables)

In [ ]:
# --- Save Global Features ---
# Rename 'ds' back to WEEK_START_DT for downstream notebook compatibility
df_save = df_features.rename(columns={'ds': 'WEEK_START_DT'})

# Convert to Spark DataFrame and write as Delta table
sdf_out = spark.createDataFrame(df_save)
sdf_out.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(OUTPUT_TABLE)
print(f"âœ… Saved global features: {OUTPUT_TABLE}")
print(f"   Shape: {df_save.shape[0]} rows Ã— {df_save.shape[1]} cols")

In [ ]:
# --- Save Per-Cluster Feature Tables ---
for cl in sorted(df_features['profile_cluster'].unique()):
    cl_df = df_features[df_features['profile_cluster'] == cl].copy()
    cl_save = cl_df.rename(columns={'ds': 'WEEK_START_DT'})
    table_name = f"supermarket_net_sales_forecast_features_cluster_{cl}"
    sdf_cl = spark.createDataFrame(cl_save)
    sdf_cl.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_name)
    print(f"  âœ… {table_name}: {cl_save.shape[0]} rows")

print(f"\nâœ… All per-cluster tables saved ({len(clusters)} clusters)")
print(f"   Total rows across all clusters: {df_features.shape[0]}")

# Summary

Feature engineering complete. Output tables ready for Notebook 06 (Train/Tune).

| Metric | Value |
|--------|-------|
| Input table | supermarket_net_sales_forecast_clustered |
| Output table | supermarket_net_sales_forecast_features |
| Total rows | 2,450 (49 per store Ã— 50 stores) |
| Total features | ~107 (numeric + lag + rolling + calendar + binary flags) |
| Clusters | 5 (erratic, regular_0, regular_1, regular_2, regular_3) |
| Top feature (global) | lag52 (0.80 importance) |
| Top correlation | lag52 (0.97) |

**Key findings:**
- `lag52` dominates globally â€” strong year-over-year seasonality
- `rolling_mean_52` and `rolling_mean_13` capture trend signals
- Per-cluster differences:
  - `erratic`: more reliance on short-term rolling features
  - `regular_1`: MARKET_SHARE_AREA_POSTAL is the top driver
  - `regular_2`: rolling_std_52 and AVG_SERVICE_GAP_RATIO are important